# 06 — Subtype-Stratified Analysis

Test whether the 25-month transition cut-point separates PAM50 breast subtypes in
a manner consistent with known proliferative biology. Subtypes are obtained from
the TCGA-BRCA PanCancer Atlas (cBioPortal) clinical file.

**Manuscript Section 3.1** reports a Spearman correlation (Ki-67 proliferative
rank vs median recurrence time) of rho = +0.60 across four subtypes (TNBC, HER2+,
Luminal B, Luminal A). With n=4 subtypes this is not statistically powered (P=0.40),
but the directional pattern corroborates the proliferation-driven temporal
transition.

**Expected input file:**
`brca_tcga_pan_can_atlas_2018_clinical_data.tsv` (cBioPortal export) placed in
`tcga_clinical/` next to TCGA-CDR.csv. Download from:
https://www.cbioportal.org/study/clinicalData?id=brca_tcga_pan_can_atlas_2018


In [ ]:
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

BASE_DIR = Path(os.environ.get("COSMOS_BASE_DIR", "./data"))
RAW_DIR = BASE_DIR / "Raw Data"
INTER_DIR = BASE_DIR / "intermediates"
DAYS_PER_MO = 365.25 / 12
T_BREAST = 25


## Load TCGA-BRCA subtypes


In [ ]:
def find_subtype_file(raw_dir):
    """Locate the BRCA PanCan Atlas clinical file by trying known paths."""
    candidates = [
        raw_dir / "tcga_clinical" / "brca_tcga_pan_can_atlas_2018_clinical_data.tsv",
        raw_dir / "brca_tcga_pan_can_atlas_2018_clinical_data.tsv",
    ]
    for p in candidates:
        if p.exists():
            return p
    # Fall back to glob search
    for pattern in ["*brca_tcga_pan_can*", "*brca*pan_can*", "*BRCA*subtype*"]:
        hits = list(raw_dir.rglob(pattern))
        if hits:
            return hits[0]
    return None


def build_subtype_map(subtype_file):
    """Parse subtype file and return {patient_barcode: subtype_label}."""
    sep = "\t" if subtype_file.suffix in (".tsv", ".txt") else ","
    sf = pd.read_csv(subtype_file, sep=sep, low_memory=False, comment="#")

    # Find patient barcode column (cBioPortal uses 'Patient ID')
    bc_col = next(
        (c for c in sf.columns
         if c.strip() in ("Patient ID", "patient_id", "Patient Identifier")
         or ("patient" in c.lower() and "id" in c.lower())),
        sf.columns[0],
    )
    sf[bc_col] = sf[bc_col].astype(str).str[:12]
    sf = sf.set_index(bc_col)

    # Find subtype column
    subtype_col = next(
        (c for c in sf.columns
         if c.strip() in ("Subtype", "SUBTYPE", "subtype",
                           "Subtype_Selected", "PAM50", "pam50")),
        None,
    )
    if subtype_col is None:
        raise ValueError(f"No subtype column found in {subtype_file.name}")

    subtype_map = {}
    for idx_val, val in sf[subtype_col].items():
        v = str(val).strip().upper()
        if "BASAL" in v or "TNBC" in v or "TRIPLE" in v:
            subtype_map[idx_val] = "TNBC"
        elif "HER2" in v:
            subtype_map[idx_val] = "HER2+"
        elif "LUMA" in v or v in ("BRCA_LUMA", "LUM_A", "LUMINAL A"):
            subtype_map[idx_val] = "Luminal A"
        elif "LUMB" in v or v in ("BRCA_LUMB", "LUM_B", "LUMINAL B"):
            subtype_map[idx_val] = "Luminal B"
        elif "LUM" in v:
            subtype_map[idx_val] = "Luminal A"
    return subtype_map


subtype_file = find_subtype_file(RAW_DIR)
if subtype_file is None:
    raise FileNotFoundError(
        "BRCA PanCan Atlas subtype file not found. Download from cBioPortal: "
        "https://www.cbioportal.org/study/clinicalData?id=brca_tcga_pan_can_atlas_2018"
    )
print(f"Subtype file: {subtype_file.name}")
subtype_map = build_subtype_map(subtype_file)
print(f"Subtypes assigned: {pd.Series(subtype_map).value_counts().to_dict()}")


## Build subtype-stratified DataFrame


In [ ]:
cdr_path = RAW_DIR / "tcga_clinical" / "TCGA-CDR.csv"
if not cdr_path.exists():
    cdr_path = cdr_path.with_suffix(".xlsx")
cdr = pd.read_csv(cdr_path) if cdr_path.suffix == ".csv" else pd.read_excel(cdr_path)


def brca_endpoint(ep):
    sub = cdr[cdr["type"] == "BRCA"].copy()
    if "bcr_patient_barcode" in sub.columns:
        sub = sub.set_index("bcr_patient_barcode")
    sub.index = sub.index.astype(str).str[:12]
    tc = f"{ep}.time"
    if tc not in sub.columns:
        return None
    out = pd.DataFrame({
        "time": pd.to_numeric(sub[tc], errors="coerce") / DAYS_PER_MO,
        "event": (pd.to_numeric(sub[ep], errors="coerce") > 0).astype(float),
    }, index=sub.index).dropna()
    return out[out["time"] > 0]


# Use DFI primary, PFI fallback (matches Cox analysis in notebook 04)
brca_clin = brca_endpoint("DFI")
ep_used = "DFI"
if brca_clin is None or brca_clin["event"].sum() < 30:
    brca_clin = brca_endpoint("PFI")
    ep_used = "PFI"
print(f"Using {ep_used} endpoint with {int(brca_clin['event'].sum())} events")

# Join clinical with subtype map
rows = []
for bc, st in subtype_map.items():
    if bc not in brca_clin.index:
        continue
    rows.append({
        "patient_id": bc,
        "subtype": st,
        "time": float(brca_clin.loc[bc, "time"]),
        "event": float(brca_clin.loc[bc, "event"]),
    })
sub_df = pd.DataFrame(rows).dropna()
print(f"Joined cohort: n={len(sub_df)}")
print(f"  Subtype counts: {sub_df['subtype'].value_counts().to_dict()}")


## Subtype-level recurrence metrics

For each PAM50 subtype, compute:
- Number of patients and recurrence events
- Median recurrence time among events
- Fraction of recurrences within the 25-month transition window


In [ ]:
def subtype_metrics(df, t, label):
    """Compute median recurrence time and percent-early for one subtype."""
    recurrers = df[df["event"] == 1]
    n_ev = len(recurrers)
    if n_ev == 0:
        return None
    return {
        "n": len(df),
        "n_events": n_ev,
        "median_rec": float(recurrers["time"].median()),
        "mean_rec": float(recurrers["time"].mean()),
        "pct_early": 100.0 * (recurrers["time"] <= t).mean(),
    }


brca_order = ["TNBC", "HER2+", "Luminal B", "Luminal A"]
ki67_rank = {"TNBC": 1, "HER2+": 2, "Luminal B": 3, "Luminal A": 4}

brca_res = {}
print(f"  {'Subtype':<12} {'n':>5} {'events':>7} {'median_m':>10} {'pct_early':>10}")
for st in brca_order:
    grp = sub_df[sub_df["subtype"] == st]
    m = subtype_metrics(grp, T_BREAST, st)
    if m and m["n_events"] >= 3:
        brca_res[st] = m
        print(f"  {st:<12} {m['n']:>5} {m['n_events']:>7} "
              f"{m['median_rec']:>9.1f}m {m['pct_early']:>9.1f}%")


## Spearman correlation: Ki-67 rank vs recurrence timing

The Ki-67 proliferative rank (1 = most proliferative TNBC, 4 = least proliferative
Luminal A) is correlated against median recurrence time and percent-early. A
positive correlation with median time (faster-growing subtypes recur earlier)
and a negative correlation with percent-early are both expected.


In [ ]:
pairs_med = [
    (ki67_rank[st], brca_res[st]["median_rec"])
    for st in brca_order if st in brca_res
]
pairs_pct = [
    (ki67_rank[st], brca_res[st]["pct_early"])
    for st in brca_order if st in brca_res
]

if len(pairs_med) >= 3:
    rho_med, p_med = spearmanr(
        [p[0] for p in pairs_med], [p[1] for p in pairs_med]
    )
    rho_pct, p_pct = spearmanr(
        [p[0] for p in pairs_pct], [p[1] for p in pairs_pct]
    )
    print(f"Spearman (Ki-67 rank vs median recurrence): "
          f"rho = {rho_med:+.4f}, P = {p_med:.4f}, n = {len(pairs_med)}")
    print(f"Spearman (Ki-67 rank vs pct early): "
          f"rho = {rho_pct:+.4f}, P = {p_pct:.4f}, n = {len(pairs_pct)}")
    print()
    print("Direction:")
    print("  rho > 0 (median): subtypes with higher Ki-67 rank (less proliferative)")
    print("                    have later median recurrence")
    print("  rho < 0 (pct):    subtypes with higher Ki-67 rank have fewer early")
    print("                    recurrences")
    print()
    print(f"With n = {len(pairs_med)} subtypes the P-value is not statistically")
    print(f"meaningful, but the directional pattern corroborates the")
    print(f"proliferation-driven temporal transition.")
else:
    rho_med = rho_pct = p_med = p_pct = None
    print(f"Only {len(pairs_med)} subtypes available; Spearman not computed.")


## Save outputs


In [ ]:
def save_pkl(obj, name):
    with open(INTER_DIR / f"{name}.pkl", "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)


subtype_results = {
    "brca_res": brca_res,
    "ki67_rank": ki67_rank,
    "endpoint_used": ep_used,
    "spearman_median": {"rho": rho_med, "p": p_med, "n": len(pairs_med)} if rho_med is not None else None,
    "spearman_pct_early": {"rho": rho_pct, "p": p_pct, "n": len(pairs_pct)} if rho_pct is not None else None,
}
save_pkl(subtype_results, "subtype_results")
print("Saved subtype_results.pkl")
